<a href="https://colab.research.google.com/github/jmishra01/AI-ML/blob/main/Building_An_Embedding_Model_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Building an Embedding Model From Scratch
========================================

This is a minimal but complete example of how real sentence-embedding models (SBERT, E5, BGE, etc.) are conceptually trained:

1. A small transformer ENCODER ( not a generative decorder) turns a sequence of tokens into a single fixed-size vector (via pooling).
2. We train it with a CONTRASTIVE LOSS: pull unrelated ones apart.
3. At inference time, one forward pass gives you one vector. Compare vectors with cosine similarity for search, clustering, dedup, etc.

This is intentionally tiny (toy vocab, toy data, few params) so it runs fast on CPU and is easy to read end-to-end. Swap in a real tokenizer, a bigger transformer, and millions of sentence pairs to get something production-grade (that's literally what SBERT-style training does).

In [1]:
import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
torch.manual_seed(0)
random.seed(0)

## Toy Data
Pairs of sequences that mean the same thing (positives). In real training this is millions of pairs (e.g. question/answer, or two paraphrases, or (query, relevant_document) pairs).

In [10]:
pairs = [
    ("how do i reset my password", "steps to recover a forgotten password"),
    ("best pizza place near me", "good pizza restaurants nearby"),
    ("what is the weather today", "current weather conditions now"),
    ("cheap flights to paris", "affordable airfare to paris"),
    ("symptoms of the flu", "signs that you have influenza"),
    ("how to bake bread at home", "homemade bread baking instructions"),
    ("best laptop for programming", "good laptops for software development")
]

Build a tiny vocabulary from the tiny corpus.

In [27]:
vocab = {"<pad>": 0, "<unk>": 1}
for a, b in pairs:
    for sent in (a, b):
        for word in sent.split():
            if word not in vocab:
                vocab[word] = len(vocab)

print(f"Vocab size: {len(vocab)}")
for k, v in list(vocab.items())[:5]:
    print(f"|{k:>14} | {v:>5} |")

Vocab size: 55
|         <pad> |     0 |
|         <unk> |     1 |
|           how |     2 |
|            do |     3 |
|             i |     4 |


In [7]:
def encode_tokens(sentence, max_len=12):
    ids = [vocab.get(w, vocab["<unk>"]) for w in sentence.split()]
    ids = ids[:max_len] + [vocab["<pad>"]] * max(0, max_len - len(ids))
    return torch.tensor(ids)

## Encoder Itself

$$ \fbox{embedding layer} + \fbox{a couble of self-attention (transformer) blocks} + \fbox{mean-pooling over tokens} → \fbox{one vector} $$


This pooled vector is the sentence embedding.

In [11]:
class TinyTransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=2, max_len=12) -> None:
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=128,
            batch_first=True,
            activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer=encoder_layer, num_layers=n_layers)
        self.out_proj = nn.Linear(d_model, d_model) # final embedding projection

    def forward(self, token_ids):
        # token_ids: (batch, seq_len)

        position = torch.arange(token_ids.size(1), device=token_ids.device)
        x = self.token_emb(token_ids) + self.pos_emb(position)

        pad_mask = token_ids == 0
        x = self.encoder(x, src_key_padding_mask=pad_mask)

        # Mean-pool over real (non-pad) tokens -> fixed-size sentence vector
        mask = (~pad_mask).unsqueeze(-1).float()
        summed = (x * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        pooled = summed / counts

        emb = self.out_proj(pooled)

        return F.normalize(emb, p=2, dim=-1)    # unit-length vectors -> cosine sim = dot product


## Contrastive loss (InfoNCE-style)
For each anchor sentence, its paired sentence should be the closest match among everything else in the batch. This is exactly how SimCSE / SBERT-style training pulls positives together and pushes negatives (everything else in the batch) apart -- no manually labeled negatives needed.

In [14]:
def contrastive_loss(emb_a, emb_b, temperature=0.05):
    # similarity matrix: every anchor vs every candidate in the batch
    logits = emb_a @ emb_b.T / temperature
    labels = torch.arange(emb_a.size(0))    # the correct match is the diagonal
    loss_a = F.cross_entropy(logits, labels)
    loss_b = F.cross_entropy(logits.T, labels)  # symmetric
    return (loss_a + loss_b) / 2


## Train

In [15]:
model = TinyTransformerEncoder(vocab_size=len(vocab))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

a_batch = torch.stack([encode_tokens(a) for a, _ in pairs])
b_batch = torch.stack([encode_tokens(b) for _, b in pairs])

print("Training tiny embedding model... \n")
for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    emb_a = model(a_batch)
    emb_b = model(b_batch)

    loss = contrastive_loss(emb_a=emb_a, emb_b=emb_b)
    loss.backward()
    optimizer.step()

    if epoch % 40 == 0 or epoch == 199:
        print(f"epoch {epoch:3d} loss {loss.item():.4f}")

Training tiny embedding model... 

epoch   0 loss 2.9487
epoch  40 loss 0.0002
epoch  80 loss 0.0001
epoch 120 loss 0.0001
epoch 160 loss 0.0000
epoch 199 loss 0.0000


## Inference
embedd new sentences and compare with cosine similarity (which is just a dot produt, since vectors are unit-normalized).

In [16]:
model.eval()

TinyTransformerEncoder(
  (token_emb): Embedding(55, 64, padding_idx=0)
  (pos_emb): Embedding(12, 64)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (out_proj): Linear(in_features=64, out_features=64, bias=True)
)

In [17]:
def embed(sentence):
    with torch.no_grad():
        return model(encode_tokens(sentence).unsqueeze(0))[0]

In [19]:
print("\n--- Similarity check on TRAINING pairs (should be high) ---")
for a,b in pairs[:3]:
    sim = (embed(a) @ embed(b)).item()
    print(f"{sim:3f} | '{a}' <-> '{b}'")


--- Similarity check on TRAINING pairs (should be high) ---
0.960131 | 'how do i reset my password' <-> 'steps to recover a forgotten password'
0.880327 | 'best pizza place near me' <-> 'good pizza restaurants nearby'
0.891621 | 'what is the weather today' <-> 'current weather conditions now'


In [20]:
print("---Similarity check: unrelated sentence vs trained ones ---")
query = "how to bake bread at home"

candidates = [b for _, b in pairs]
q_emb = embed(query)

sims = [(c, (q_emb @ embed(c)).item()) for c in candidates]
sims.sort(key=lambda x: -x[1])
print(f"Query: '{query}'")
for c, sim in sims[:3]:
    print(f"{sim:3f} | '{c}'")

---Similarity check: unrelated sentence vs trained ones ---
Query: 'how to bake bread at home'
0.889960 | 'homemade bread baking instructions'
0.362046 | 'steps to recover a forgotten password'
0.335232 | 'signs that you have influenza'
